# Exemplo: Trace ReAct Passo a Passo
## Aula 10 — Agentic RAG | MBA RAG & CAG Aplicados a Direito e Segurança Pública

Este notebook demonstra **como escrever boas descrições de ferramentas** para agentes RAG jurídicos.

---

### Referência
YAO, Shunyu et al. **ReAct: Synergizing Reasoning and Acting in Language Models**. ICLR, 2023.

In [ ]:
# Comparação: descrições ruins vs boas de ferramentas
# O LLM usa a descrição para decidir QUANDO e COMO usar cada ferramenta

descricoes_ruins_vs_boas = [
    {
        "ferramenta": "search_docs",
        "ruim": "Busca documentos.",
        "boa": (
            "Busca legislação, jurisprudência e doutrina jurídica no banco vetorial local.\n"
            "USE QUANDO: a pergunta envolver leis, artigos de lei, decisões judiciais, "
            "acórdãos, institutos jurídicos ou procedimentos processuais.\n"
            "NÃO USE PARA: notícias recentes, dados estatísticos ou fatos externos ao direito.\n"
            "ARGS: query (str) - termos de busca em português, descrevendo o tema jurídico"
        ),
        "motivo": "A descrição ruim não diz QUANDO usar nem o que esperar como retorno."
    },
    {
        "ferramenta": "query_db",
        "ruim": "Executa uma query no banco de dados.",
        "boa": (
            "Executa consultas SQL no banco de dados de ocorrências e legislação.\n"
            "TABELAS: acordaos (numero, tribunal, data, relator, ementa, resultado), "
            "ocorrencias (numero_bo, data, tipo_crime, municipio, status), "
            "legislacao (numero, tipo, ementa, artigos_principais)\n"
            "USE QUANDO: precisar de dados estruturados, filtros por data/tipo, ou contagens.\n"
            "NÃO USE: para busca semântica (use search_docs). APENAS SELECT é permitido.\n"
            "ARGS: sql_query (str) - query SQL válida começando com SELECT"
        ),
        "motivo": "Sem saber as tabelas e colunas, o LLM não consegue escrever SQL correto."
    },
    {
        "ferramenta": "web_search",
        "ruim": "Busca na internet.",
        "boa": (
            "Busca informações atuais na internet sobre tópicos jurídicos e segurança pública.\n"
            "USE QUANDO: precisar de notícias recentes (últimos 2 anos), análises acadêmicas "
            "recentes, mudanças legislativas ou dados que não estão na base local.\n"
            "NÃO USE: se a informação provavelmente está no banco local (evite redundância).\n"
            "USE APÓS: search_docs e query_db, como complemento — não como primeira opção.\n"
            "ARGS: query (str) - termos de busca em português"
        ),
        "motivo": "Sem indicar ordem de prioridade, o LLM pode usar web antes do banco local."
    }
]

print("📝 COMPARAÇÃO: Descrições de Ferramentas — Ruim vs Bom")
print("=" * 70)

for item in descricoes_ruins_vs_boas:
    print(f"\n🔧 FERRAMENTA: {item['ferramenta']}")
    print(f"\n  ❌ RUIM:")
    print(f"     '{item['ruim']}'")
    print(f"\n  ✅ BOA:")
    for linha in item['boa'].split('\n'):
        print(f"     {linha}")
    print(f"\n  💡 POR QUÊ IMPORTA:")
    print(f"     {item['motivo']}")
    print("-" * 70)

In [ ]:
# Simulação completa de um trace ReAct jurídico
# Mostra o que o LangGraph executa internamente

from dataclasses import dataclass, field
from typing import List, Optional
import time

@dataclass
class ReActStep:
    step_num: int
    thought: str
    action: Optional[str] = None
    action_input: Optional[dict] = None
    observation: Optional[str] = None
    is_final: bool = False
    answer: Optional[str] = None
    latencia_ms: int = 0


# Trace completo para uma pergunta complexa sobre lavagem de dinheiro
trace_lavagem = [
    ReActStep(
        step_num=1,
        thought=(
            "A pergunta é sobre precedentes do STJ e STF em lavagem de dinheiro via cripto em 2023-2024. "
            "Vou começar buscando jurisprudência local — é mais rápido que web search."
        ),
        action="search_docs",
        action_input={"query": "lavagem dinheiro criptomoeda STJ STF 2023 2024"},
        observation=(
            "Encontrados 2 acórdãos:\n"
            "  📋 RHC 150234 (STJ, 2023-08-22): Lavagem de dinheiro. Prisão preventiva. Resultado: Provido.\n"
            "  📋 AP 996 (STF, 2023-12-01): Lavagem transnacional. Condenação 12 anos."
        ),
        latencia_ms=320
    ),
    ReActStep(
        step_num=2,
        thought=(
            "Tenho 2 acórdãos. Preciso verificar se há ocorrências de lavagem via cripto "
            "no banco de dados para contexto factual."
        ),
        action="query_db",
        action_input={"sql_query": "SELECT numero_bo, data, descricao FROM ocorrencias WHERE LOWER(descricao) LIKE '%cripto%' OR LOWER(descricao) LIKE '%criptomoeda%'"},
        observation=(
            "1 registro | BO-2024-011 | 2024-02-03 | "
            "Suspeito utilizou criptomoedas para dissimular origem de R$ 2,3M provenientes de tráfico."
        ),
        latencia_ms=45
    ),
    ReActStep(
        step_num=3,
        thought=(
            "Tenho jurisprudência e dado factual. Falta verificar a legislação mais atualizada "
            "— o COAF tem emitido normas sobre cripto. Farei busca web para informações mais recentes."
        ),
        action="web_search",
        action_input={"query": "COAF regulação criptomoedas lavagem dinheiro Brasil 2024 legislação"},
        observation=(
            "[Simulação] 📰 'Receita Federal amplia obrigações de reporte de exchanges' — Valor Econômico, jan/2024\n"
            "[Simulação] 📰 'COAF identifica 40% mais casos de cripto suspeita em 2023' — Folha, fev/2024"
        ),
        latencia_ms=2100
    ),
    ReActStep(
        step_num=4,
        thought=(
            "Agora tenho: 2 precedentes judiciais, 1 caso factual e dados legislativos recentes. "
            "Tenho informação suficiente para elaborar uma resposta completa e fundamentada."
        ),
        is_final=True,
        answer=(
            "**Precedentes sobre Lavagem de Dinheiro via Criptomoedas (2023-2024)**\n\n"
            "**Jurisprudência relevante:**\n"
            "1. **RHC 150234/STJ** (ago/2023): O STJ proveu o recurso, exigindo fundamentação "
            "específica para prisão preventiva em casos de lavagem, mesmo envolvendo cripto.\n"
            "2. **AP 996/STF** (dez/2023): Condenação a 12 anos por lavagem transnacional. "
            "Caso envolveu organização criminosa com operações em múltiplos países.\n\n"
            "**Dado factual:** BO-2024-011 (RJ/fev-2024): Caso registrado de uso de cripto "
            "para dissimular R$ 2,3M oriundos de tráfico.\n\n"
            "**Contexto legislativo:** A Receita Federal e COAF ampliaram as obrigações de "
            "reporte das exchanges em 2024, com aumento de 40% na identificação de casos suspeitos."
        ),
        latencia_ms=4200
    )
]


# Exibe o trace formatado
total_tokens_estimados = 0
total_ms = 0

print("🤖 TRACE REACT COMPLETO — Lavagem via Criptomoedas")
print("=" * 70)
print("❓ Pergunta: Quais os precedentes e dados sobre lavagem de dinheiro via cripto?")
print("=" * 70)

for step in trace_lavagem:
    total_ms += step.latencia_ms
    print(f"\n{'─'*60}")
    print(f"📍 PASSO {step.step_num} ({step.latencia_ms}ms)")
    print(f"{'─'*60}")
    print(f"🧠 THOUGHT:")
    print(f"   {step.thought}")
    
    if step.action:
        print(f"\n⚡ ACTION: {step.action}")
        print(f"   Input: {step.action_input}")
    
    if step.observation:
        print(f"\n👁️  OBSERVATION:")
        for linha in step.observation.split("\n"):
            print(f"   {linha}")
    
    if step.is_final:
        print(f"\n✅ RESPOSTA FINAL:")
        for linha in step.answer.split("\n"):
            print(f"   {linha}")

print(f"\n{'='*70}")
print(f"📊 MÉTRICAS DO TRACE:")
print(f"   Total de steps: {len(trace_lavagem)}")
print(f"   Total de tool calls: {sum(1 for s in trace_lavagem if s.action)}")
print(f"   Latência total: {total_ms}ms ({total_ms/1000:.1f}s)")
print(f"   Ferramentas usadas: {[s.action for s in trace_lavagem if s.action]}")
print(f"   Web search: {sum(1 for s in trace_lavagem if s.action == 'web_search')}x (a mais lenta: 2100ms)")

---
## Análise: Padrões de Eficiência em Traces ReAct

| Padrão | Sinal | Diagnóstico | Solução |
|--------|-------|-------------|----------|
| Web search antes de search_docs | ⚠️ | Ordem ineficiente | Ajustar prioridade no prompt |
| Mesma tool 2x com query similar | ⚠️ | Redundância | Adicionar memória de observações |
| >6 tool calls para query simples | ❌ | Agente perdido | Revisar system prompt |
| Thought vago sem ação clara | ❌ | Raciocínio fraco | Prompt com exemplos (few-shot) |
| Answer sem citar fontes | ❌ | Baixa faithfulness | Exigir citações no prompt |